# Part 5 — Out-of-Sample Replication on New Symbols
Part 4's pre-registered test passed in thin books, but referees will ask whether it holds **outside the 30 symbols where it was found**. This notebook answers that on symbols never used before, with the **same frozen specification**.

| Cell | What it does |
|---|---|
| Y0–Y1 | setup and the same functions as Part 4 (shares the Drive cache) |
| Y2 | picks up to 24 **new** thin-volume symbols, seeded, before any analysis |
| Y3 | downloads and validates them (rules V1–V5) |
| Y4 | thin classification (fixed cut 0.39 $M) and hourly blocks, plus a **unit-free amplification ratio** and **the three terms of C3** |
| Y5 | **new pre-registration**: commit `prereg/P5_replication.json` before Y6 |
| Y6 | **the replication test: run ONCE** |
| Y7 | training-only descriptives and the download |

**Rules:** run cells one at a time (no "Run all"). Run Y6 exactly once. Whatever Y6 shows goes in the paper.
Expected time: 30–90 minutes (downloads).

In [ ]:
# ============================================================================
# CELL Y0 — SETUP (self-contained; run first)
# Part 5: out-of-sample REPLICATION of the thin-book result on symbols never used before,
# plus a unit-free amplification ratio and a decomposition of C3 into its three terms.
# Results go to the same results/ folder as Parts 1-2 when USE_DRIVE = True.
# ============================================================================
import os, sys, json, time, math, glob, re, io, zipfile, hashlib, subprocess, warnings, zlib
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed
import xml.etree.ElementTree as ET

IN_COLAB = "google.colab" in sys.modules
try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
import numpy as np
import pandas as pd
from scipy import stats, optimize
warnings.filterwarnings("ignore", category=RuntimeWarning)

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ceiling_suite"
elif IN_COLAB:
    BASE = "/content/ceiling_suite"
else:
    BASE = os.path.abspath("ceiling_suite_local")
RESULTS, PREREG = os.path.join(BASE, "results"), os.path.join(BASE, "prereg")
XDATA = os.path.join(BASE, "data_xsec")
for _d in (RESULTS, PREREG, XDATA):
    os.makedirs(_d, exist_ok=True)

SEED = 20261020
def rng(offset=0):
    return np.random.default_rng(SEED + offset)

def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

def record(test_id, passed, kind, metrics, criterion="", note=""):
    assert kind in {"theorem-check", "simulation", "real-data", "audit"}, kind
    status = "REPORT" if passed is None else ("PASS" if passed else "FAIL")
    out = {"test_id": test_id, "status": status, "kind": kind, "criterion": criterion, "note": note,
           "metrics": _clean(metrics), "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "seed": SEED}
    with open(os.path.join(RESULTS, f"{test_id}.json"), "w") as f:
        json.dump(out, f, indent=2)
    print(f"[{status}] {test_id} ({kind})" + (f" — {criterion}" if criterion else ""))
    if note:
        print("        note:", note)
    return out

def boot_ci(v, R, B=2000, stat=np.mean):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 3:
        return [np.nan, np.nan]
    bs = [stat(v[R.integers(0, len(v), len(v))]) for _ in range(B)]
    return [float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))]

# ---- Sample design (fixed before looking at any Part 4 data) ----------------
N_SYMBOLS = 30                  # symbols in the cross-section, spread across liquidity (BTC and ETH always in)
PANEL_DAYS_OF_MONTH = (10, 24)  # same panel design as Part 1
PANEL_START, PANEL_END = "2023-01-01", "2026-09-30"
TRAIN_END, TEST_START = "2024-12-31", "2025-01-01"
LIQ_RANK_DATE = "2024-06-10"    # liquidity ranking day for choosing symbols (training period)
THREADS = 12
ROOT = "https://data.binance.vision/data/futures/um/daily"
print("BASE =", BASE)

def _http(url, method="GET", timeout=60, tries=3):
    last = None
    for i in range(tries):
        try:
            req = urllib.request.Request(url, method=method, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.status, (r.read() if method == "GET" else b"")
        except urllib.error.HTTPError as e:
            if e.code in (403, 404):
                return e.code, b""
            last = e
        except Exception as e:
            last = e
        time.sleep(1.5 * (i + 1))
    raise RuntimeError(f"network error {url}: {last}")

def ds_url(ds, sym, date):
    if ds == "klines":
        return f"{ROOT}/klines/{sym}/1m/{sym}-1m-{date}.zip"
    return f"{ROOT}/{ds}/{sym}/{sym}-{ds}-{date}.zip"

def exists(ds, sym, date):
    return _http(ds_url(ds, sym, date), "HEAD", timeout=20)[0] == 200

DL_LOG = []
def fetch(ds, sym, date):
    """Zip bytes, checked against Binance's published .CHECKSUM when available; None if missing."""
    url = ds_url(ds, sym, date)
    code, raw = _http(url, timeout=180)
    if code != 200:
        return None
    sha = hashlib.sha256(raw).hexdigest()
    c2, chk = _http(url + ".CHECKSUM", timeout=30)
    verified = None
    if c2 == 200 and chk:
        if chk.decode().split()[0] != sha:
            raise RuntimeError(f"SHA-256 mismatch {url}")
        verified = True
    DL_LOG.append({"ds": ds, "symbol": sym, "date": date, "sha256": sha, "checksum_verified": verified})
    return raw

def read_zip_csv(raw, default_cols=None):
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        name = [n for n in z.namelist() if n.endswith(".csv")][0]
        txt = z.read(name).decode("utf-8", errors="ignore")
    first = txt.split("\n", 1)[0]
    has_header = bool(re.search(r"[A-Za-z_]", first.split(",")[0]))
    df = pd.read_csv(io.StringIO(txt), header=0 if has_header else None)
    if not has_header and default_cols is not None:
        df.columns = default_cols[:df.shape[1]]
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

def to_utc_seconds(col):
    s = pd.Series(col)
    if pd.api.types.is_numeric_dtype(s):
        v = s.astype("int64").to_numpy()
        return np.where(v > 1e14, v // 1_000_000, np.where(v > 1e11, v // 1000, v))
    ts = pd.to_datetime(s, utc=True)
    return ((ts - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta("1s")).astype("int64").to_numpy()

def panel_dates(start=PANEL_START, end=PANEL_END):
    ds = pd.date_range(start, end, freq="D")
    return [d.strftime("%Y-%m-%d") for d in ds if d.day in PANEL_DAYS_OF_MONTH]


In [ ]:
# ============================================================================
# CELL Y1 — FUNCTIONS (same code as Part 4; nothing is downloaded or estimated here)
# ============================================================================
FALLBACK = ("BTCUSDT ETHUSDT BNBUSDT XRPUSDT ADAUSDT DOGEUSDT SOLUSDT DOTUSDT LTCUSDT LINKUSDT BCHUSDT TRXUSDT "
            "AVAXUSDT ATOMUSDT ETCUSDT FILUSDT NEARUSDT AAVEUSDT UNIUSDT SANDUSDT MANAUSDT AXSUSDT XLMUSDT ALGOUSDT "
            "VETUSDT EOSUSDT XTZUSDT THETAUSDT ICPUSDT EGLDUSDT CHZUSDT ENJUSDT ZILUSDT KAVAUSDT RUNEUSDT GALAUSDT "
            "CRVUSDT DYDXUSDT 1INCHUSDT COMPUSDT SNXUSDT ZECUSDT DASHUSDT NEOUSDT IOTAUSDT ONTUSDT QTUMUSDT BATUSDT "
            "ZRXUSDT KNCUSDT").split()
def list_bucket_symbols(ds="bookDepth"):
    url0 = ("https://s3-ap-northeast-1.amazonaws.com/data.binance.vision?delimiter=/"
            f"&prefix=data/futures/um/daily/{ds}/")
    syms, marker = [], ""
    for _ in range(20):
        code, raw = _http(url0 + (f"&marker={marker}" if marker else ""), timeout=60)
        if code != 200 or not raw:
            return []
        root = ET.fromstring(raw)
        ns = {"s": root.tag.split("}")[0].strip("{")} if root.tag.startswith("{") else {}
        pref = [e.text for e in (root.findall(".//s:CommonPrefixes/s:Prefix", ns) if ns else root.findall(".//CommonPrefixes/Prefix"))]
        syms += [p.rstrip("/").split("/")[-1] for p in pref]
        trunc = (root.find("s:IsTruncated", ns) if ns else root.find("IsTruncated"))
        if trunc is None or trunc.text != "true" or not pref:
            break
        marker = pref[-1]
    return sorted(set(syms))
EARLY, LATE = "2023-03-10", "2026-06-10"
def span_ok(s):
    return exists("bookDepth", s, EARLY) and exists("bookDepth", s, LATE)
KL_COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time", "quote_volume",
           "count", "taker_buy_volume", "taker_buy_quote_volume", "ignore"]
def day_quote_volume(s):
    raw = fetch("klines", s, LIQ_RANK_DATE)
    return np.nan if raw is None else float(read_zip_csv(raw, KL_COLS)["quote_volume"].astype(float).sum())
NEED_BANDS = [-5, -2, -1, 1, 2, 5]
def nice_tick(x):
    """Snap a positive increment to the nearest 1, 2 or 5 x 10^k (exchange tick grids)."""
    if not np.isfinite(x) or x <= 0:
        return np.nan
    e = math.floor(math.log10(x)); m = x / 10 ** e
    c = min([1, 2, 5, 10], key=lambda v: abs(math.log(m / v)))
    return c * 10 ** e
def infer_tick(k):
    v = np.unique(np.round(k[["open", "high", "low", "close"]].astype(float).to_numpy().ravel(), 10))
    dv = np.diff(v); dv = dv[dv > 0]
    return nice_tick(np.quantile(dv, 0.02)) if len(dv) > 5 else np.nan
def validated_snapshots(sym, date, close_by_minute, t0):
    """Validated bookDepth snapshots: DataFrame indexed by second with bid/ask notional at 1,2,5% ($M)."""
    raw = fetch("bookDepth", sym, date)
    if raw is None:
        return None, {"snapshots": 0}
    b = read_zip_csv(raw)
    b["sec"] = to_utc_seconds(b["timestamp"])
    b["percentage"] = pd.to_numeric(b["percentage"], errors="coerce").round().astype("Int64")
    piv_n = b.pivot_table(index="sec", columns="percentage", values="notional", aggfunc="last")
    piv_d = b.pivot_table(index="sec", columns="percentage", values="depth", aggfunc="last")
    n0 = len(piv_n)
    ok = piv_n.reindex(columns=NEED_BANDS).notna().all(axis=1) & piv_d.reindex(columns=NEED_BANDS).notna().all(axis=1)
    piv_n, piv_d = piv_n[ok], piv_d[ok]
    mono = (piv_d[-2] >= piv_d[-1]) & (piv_d[-5] >= piv_d[-2]) & (piv_d[2] >= piv_d[1]) & (piv_d[5] >= piv_d[2])
    minute = ((piv_n.index.to_numpy() - t0) // 60).clip(0, 1439)
    ref = close_by_minute[minute]
    imp_b, imp_a = piv_n[-1] / piv_d[-1], piv_n[1] / piv_d[1]
    price_ok = (np.abs(imp_b / ref - 1) < 0.03) & (np.abs(imp_a / ref - 1) < 0.03)
    sig = piv_n[NEED_BANDS].round(2).astype(str).agg("|".join, axis=1)
    runlen = sig.groupby((sig != sig.shift()).cumsum()).transform("size")
    not_stale = runlen <= 10
    good = mono & price_ok & not_stale
    g = piv_n[good]
    snap = pd.DataFrame({"bid1": g[-1], "ask1": g[1], "bid2": g[-2], "ask2": g[2], "bid5": g[-5], "ask5": g[5]}) / 1e6
    st = {"snapshots": n0, "drop_V1": int((~ok).sum()), "drop_V2": int((~mono).sum()),
          "drop_V3": int((~price_ok).sum()), "drop_V4": int((~not_stale).sum())}
    return snap, st
def build_day(sym, date):
    t0 = int(pd.Timestamp(f"{date} 00:00:00", tz="UTC").timestamp())
    idx = np.arange(1440)
    raw = fetch("klines", sym, date)
    if raw is None:
        return None, {"symbol": sym, "date": date, "reason": "no klines"}
    k = read_zip_csv(raw, KL_COLS)
    tick = infer_tick(k)
    k = k.assign(m=(to_utc_seconds(k["open_time"]) - t0) // 60).drop_duplicates("m").set_index("m").reindex(idx)
    out = pd.DataFrame(index=idx)
    out["close"] = k["close"].astype(float).ffill().bfill()
    out["flow"] = (2 * k["taker_buy_quote_volume"].astype(float) - k["quote_volume"].astype(float)).fillna(0) / 1e6
    out["vol"] = k["quote_volume"].astype(float).fillna(0) / 1e6
    st = {"symbol": sym, "date": date, "tick": tick}
    snap, s2 = validated_snapshots(sym, date, out["close"].to_numpy(), t0)
    st.update(s2)
    for c in ["bid1", "ask1", "bid2", "ask2", "bid5", "ask5"]:
        out[c] = np.nan
    if snap is not None and len(snap):
        mm = ((snap.index.to_numpy() - t0) // 60).clip(0, 1439)
        sm = snap.assign(m=mm).groupby("m").last().reindex(idx).ffill(limit=2)
        for c in snap.columns:
            out[c] = sm[c].to_numpy()
    st["minute_coverage"] = float(out["bid1"].notna().mean())
    st["depth_day_ok"] = st["minute_coverage"] >= 0.80
    for c in ["oi", "top_pos_ratio", "top_acc_ratio", "acc_ratio", "taker_ratio"]:
        out[c] = np.nan
    raw = fetch("metrics", sym, date)
    if raw is not None:
        mt = read_zip_csv(raw)
        tcol = "create_time" if "create_time" in mt.columns else mt.columns[0]
        mm = ((to_utc_seconds(mt[tcol]) - t0) // 60).clip(0, 1439)
        for c, src, scale in [("oi", "sum_open_interest_value", 1e6),
                              ("top_pos_ratio", "sum_toptrader_long_short_ratio", 1.0),
                              ("top_acc_ratio", "count_toptrader_long_short_ratio", 1.0),
                              ("acc_ratio", "count_long_short_ratio", 1.0),
                              ("taker_ratio", "sum_taker_long_short_vol_ratio", 1.0)]:
            if src in mt.columns:
                s = pd.Series(pd.to_numeric(mt[src], errors="coerce").to_numpy() / scale, index=mm)
                out[c] = s.groupby(level=0).last().reindex(idx).to_numpy()   # 5-min values, not filled
    out["symbol"], out["date"] = sym, date
    return out, st
MIN_DIR = os.path.join(XDATA, "minute")
def get_day(sym, date):
    d_ = os.path.join(MIN_DIR, sym); os.makedirs(d_, exist_ok=True)
    pq, js = os.path.join(d_, f"{date}.parquet"), os.path.join(d_, f"{date}.json")
    if os.path.exists(js):
        st = json.load(open(js))
        return (pd.read_parquet(pq) if os.path.exists(pq) else None), st
    df, st = build_day(sym, date)
    if df is not None:
        df.to_parquet(pq)
    json.dump(_clean(st), open(js, "w"))
    return df, st
def prep(df, flow_scale):
    p = np.log(df["close"].to_numpy()) * 1e4
    u = df["flow"].to_numpy() / flow_scale
    tot = (df["bid1"] + df["ask1"]).to_numpy()
    d = ((df["bid1"] - df["ask1"]) / (df["bid1"] + df["ask1"])).to_numpy()
    x = p - pd.Series(p).ewm(span=60, adjust=False).mean().to_numpy()
    return p, u, d, x, tot
def ols(y, X):
    ok = np.all(np.isfinite(X), 1) & np.isfinite(y)
    if ok.sum() < X.shape[1] + 20:
        return None
    return np.linalg.lstsq(np.column_stack([np.ones(ok.sum()), X[ok]]), y[ok], rcond=None)[0][1:]
def impulse_t6(a, phi, kappa, r, eps, eta, q, L=120):
    x = np.zeros(L); dd = np.zeros(L); f = np.zeros(L)
    for t in range(L):
        if t > 0:
            f[t] = (1 - eta) * f[t - 1] + eta * q * x[t - 1]
        tot = (1.0 if t == 0 else 0.0) + f[t]
        x[t] = (a * x[t - 1] + phi * dd[t - 1] if t else 0) + kappa * tot
        dd[t] = ((1 - r) * dd[t - 1] if t else 0) + eps * tot
        if abs(x[t]) > 1e9:
            return np.full(L, np.nan)
    return x
def fit_t6(p, u, d, x):
    dp = np.diff(p, prepend=np.nan)
    bp = ols(dp[1:], np.column_stack([u[1:], d[:-1], x[:-1]]))
    bd = ols(d[1:], np.column_stack([d[:-1], u[1:]]))
    bm = ols(u[1:], np.column_stack([u[:-1], x[:-1]]))
    if bp is None or bd is None or bm is None:
        return None
    kappa, phi, a = bp[0], bp[1], 1 + bp[2]
    r, eps, eta = 1 - bd[0], bd[1], 1 - bm[0]
    q = bm[1] / eta if abs(eta) > 1e-6 else np.nan
    h = impulse_t6(a, phi, kappa, r, eps, eta, q)
    ok = np.all(np.isfinite(h)) and abs(h[0]) > 1e-12
    return {"kappa": kappa, "phi": phi, "a": a, "r": r, "eps": eps, "eta": eta, "q": q,
            "C2": phi * eps - (1 - a) * kappa, "C3": phi * eps + kappa ** 2 * eta * q - (1 - a) * kappa,
            "peak_ratio": float(np.max(h) / h[0]) if ok else np.nan, "peak_minute": int(np.argmax(h)) if ok else -1}
def loop_gain(p, u, span=30):
    x = p - pd.Series(p).ewm(span=span, adjust=False).mean().to_numpy()
    f = np.roll(u, -1)[:-1]; x = x[:-1]
    Z = np.column_stack([x, f])
    Y, X = Z[1:], np.column_stack([np.ones(len(Z) - 1), Z[:-1]])
    ok = np.all(np.isfinite(Y), 1) & np.all(np.isfinite(X), 1)
    if ok.sum() < 100:
        return np.nan
    J = np.linalg.lstsq(X[ok], Y[ok], rcond=None)[0][1:].T
    return float(J[0, 1] * J[1, 0] / (1 - J[1, 1])) if abs(1 - J[1, 1]) > 1e-9 else np.nan
def auc_(y, s):
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    rk = stats.rankdata(np.concatenate([pos, neg]))
    return (rk[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))
def cw_test(y, fb, ff, lags=6):
    adj = (y - fb) ** 2 - ((y - ff) ** 2 - (fb - ff) ** 2)
    m = adj.mean(); e = adj - m; n = len(adj)
    lrv = e @ e / n + 2 * sum((1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n for l in range(1, lags + 1))
    t_ = m / np.sqrt(lrv / n)
    return float(t_), float(1 - stats.norm.cdf(t_))
def design(D, cols, syms):
    X = [np.ones(len(D))] + [D[c].to_numpy() for c in cols] + [(D["symbol"] == s).to_numpy(float) for s in syms[1:]]
    return np.column_stack(X)
def evaluate(sub, outcome, cand):
    A0 = sub[sub["split"] == "train"].copy(); B0 = sub[sub["split"] == "test"].copy()
    lo, hi = A0[cand].quantile([0.01, 0.99]); A0["cand_w"] = A0[cand].clip(lo, hi); B0["cand_w"] = B0[cand].clip(lo, hi)
    need = CTRL + ["cand_w", outcome]
    A = A0.dropna(subset=need); B = B0.dropna(subset=need).sort_values(["date", "symbol", "hour"])
    if len(A) < 200 or len(B) < 200 or A[outcome].std() < 1e-8 or A["cand_w"].std() < 1e-8:
        return {"skipped": "too few blocks or no variation in outcome/candidate", "n_train": len(A), "n_test": len(B),
                "clark_west_p": np.nan, "delta_r2": np.nan, "delta_r2_ci": [np.nan, np.nan]}
    B = B[B["symbol"].isin(set(A["symbol"]))]
    syms = sorted(A["symbol"].unique()); y = B[outcome].to_numpy()
    bb = np.linalg.lstsq(design(A, CTRL, syms), A[outcome].to_numpy(), rcond=None)[0]
    bf = np.linalg.lstsq(design(A, CTRL + ["cand_w"], syms), A[outcome].to_numpy(), rcond=None)[0]
    fb = design(B, CTRL, syms) @ bb; ff = design(B, CTRL + ["cand_w"], syms) @ bf
    ybar = A[outcome].mean(); sst = np.sum((y - ybar) ** 2)
    t_, p_ = cw_test(y, fb, ff)
    tail = (y > A[outcome].quantile(0.9)).astype(int)
    Rb = rng(88); dd = B["date"].to_numpy(); ud = np.unique(dd)
    pos = {d: np.where(dd == d)[0] for d in ud}; dR2, dA = [], []
    for _ in range(2000):
        idx = np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))])
        s_ = np.sum((y[idx] - ybar) ** 2)
        dR2.append((np.sum((y[idx] - fb[idx]) ** 2) - np.sum((y[idx] - ff[idx]) ** 2)) / s_)
        dA.append(auc_(tail[idx], ff[idx]) - auc_(tail[idx], fb[idx]))
    return {"n_train": len(A), "n_test": len(B), "n_symbols": len(syms),
            "r2_base": 1 - np.sum((y - fb) ** 2) / sst, "r2_full": 1 - np.sum((y - ff) ** 2) / sst,
            "delta_r2": (np.sum((y - fb) ** 2) - np.sum((y - ff) ** 2)) / sst,
            "delta_r2_ci": [float(np.nanquantile(dR2, .025)), float(np.nanquantile(dR2, .975))],
            "auc_base": auc_(tail, fb), "auc_full": auc_(tail, ff),
            "delta_auc_ci": [float(np.nanquantile(dA, .025)), float(np.nanquantile(dA, .975))],
            "clark_west_t": t_, "clark_west_p": p_, "coef_candidate_train": float(bf[len(CTRL) + 1])}


## Choose and build the new symbols

In [ ]:
# ============================================================================
# CELL Y2 — CHOOSE NEW SYMBOLS (never used in Part 4), FIXED BEFORE ANY ANALYSIS
# Candidates: USD-M perpetuals with depth in both 2023 and 2026, NOT among Part 4's 30 symbols,
# whose quote volume on the Part 4 ranking day is at most 1.5x the largest volume among
# Part 4's thin-tercile symbols (so they are thin-book candidates). Up to NEW_MAX are drawn at
# random with a fixed seed. Thin status is decided later from training depth (cell Y4).
# ============================================================================
NEW_MAX = 24
P4 = json.load(open(os.path.join(RESULTS, "X1_universe.json")))["metrics"]
P4_SYMBOLS = P4["symbols"]
P4_TERC = {s: v["tercile"] for s, v in json.load(open(os.path.join(RESULTS, "X3_T6_cross_section.json")))["metrics"]["by_symbol"].items()}
qv4 = P4["quote_volume_musd_on_rank_date"]
thin_vol_max = max(qv4[s] for s in P4_SYMBOLS if P4_TERC.get(s) == "thin")
print(f"Part 4 thin symbols: max quote volume on {LIQ_RANK_DATE} = {thin_vol_max:.1f} $M")

try:
    listed = list_bucket_symbols()
except Exception as e:
    print("bucket listing failed:", e); listed = []
cands = [s for s in (listed or FALLBACK) if re.fullmatch(r"[A-Z0-9]+USDT", s) and s not in P4_SYMBOLS]
with ThreadPoolExecutor(THREADS) as ex:
    keep = [s for s, ok in zip(cands, ex.map(span_ok, cands)) if ok]
with ThreadPoolExecutor(THREADS) as ex:
    qv = dict(zip(keep, ex.map(day_quote_volume, keep)))
pool = sorted([s for s in keep if np.isfinite(qv[s]) and qv[s] / 1e6 <= 1.5 * thin_vol_max])
R_sel = rng(5)
SYMBOLS = sorted(R_sel.choice(pool, size=min(NEW_MAX, len(pool)), replace=False).tolist()) if pool else []
print(f"new candidates: {len(cands)}; with depth span: {len(keep)}; thin-volume pool: {len(pool)}; chosen: {len(SYMBOLS)}")
print(SYMBOLS)
record("Y1_new_universe", None, "audit",
       {"part4_symbols_excluded": P4_SYMBOLS, "thin_volume_cap_musd": 1.5 * thin_vol_max, "pool": pool,
        "symbols": SYMBOLS, "quote_volume_musd": {s: qv[s] / 1e6 for s in SYMBOLS}, "seed": SEED},
       note="New symbols fixed before any estimation. None was used in Part 4.")


In [ ]:
# ============================================================================
# CELL Y3 — DOWNLOAD AND VALIDATE THE NEW SYMBOLS (same rules V1-V5 as Part 4; cached in Drive)
# ============================================================================
PDATES = panel_dates()
jobs = [(s, d) for s in SYMBOLS for d in PDATES]
print(f"building {len(jobs)} symbol-days; cached ones are skipped")
PANEL, VST = {}, []
with ThreadPoolExecutor(THREADS) as ex:
    futs = {ex.submit(get_day, s, d): (s, d) for s, d in jobs}
    for i, f in enumerate(as_completed(futs), 1):
        s, d = futs[f]
        try:
            df, st = f.result(); VST.append(st)
            if df is not None:
                PANEL[(s, d)] = df
        except Exception as e:
            VST.append({"symbol": s, "date": d, "reason": f"error: {e}"})
        if i % 200 == 0:
            print(f"   {i}/{len(jobs)}")
vst = pd.DataFrame(VST)
vst["depth_day_ok"] = vst.get("depth_day_ok", pd.Series(False, index=vst.index)).fillna(False).astype(bool)
OK = sorted([(r.symbol, r.date) for r in vst.itertuples() if r.depth_day_ok and (r.symbol, r.date) in PANEL])
pd.DataFrame(DL_LOG).to_csv(os.path.join(RESULTS, "p5_manifest_sha256.csv"), index=False)
record("Y2_build", None, "audit", {"symbol_days_requested": len(jobs), "symbol_days_valid": len(OK),
       "by_symbol": vst.groupby("symbol")["depth_day_ok"].agg(["sum", "count"]).to_dict("index")})


In [ ]:
# ============================================================================
# CELL Y4 — THIN CLASSIFICATION AND HOURLY BLOCKS
# Thin = median training-period depth within 1% of mid below THIN_DEPTH_MUSD, the boundary between
# Part 4's thin and mid terciles (largest thin 0.3815 $M, smallest mid 0.4003 $M). Fixed here.
# Each block also stores the three terms of C3 and a unit-free amplification ratio:
#   A_ratio = (φε + κ²ηq) / ((1-a)κ) - 1      (> 0 exactly when C3 > 0; invariant to flow units)
# ============================================================================
THIN_DEPTH_MUSD = 0.39
FLOW_SCALE, DEPTH_TR = {}, {}
for s in SYMBOLS:
    tr_days = [d for (s2, d) in OK if s2 == s and d <= TRAIN_END]
    v = [np.nanmedian(np.abs(PANEL[(s, d)]["flow"].to_numpy())) for d in tr_days]
    FLOW_SCALE[s] = float(np.nanmedian(v)) if v and np.nanmedian(v) > 0 else 1.0
    dep = [np.nanmedian((PANEL[(s, d)]["bid1"] + PANEL[(s, d)]["ask1"]).to_numpy()) for d in tr_days]
    DEPTH_TR[s] = float(np.nanmedian(dep)) if dep else np.nan
GROUP = {s: ("thin" if DEPTH_TR[s] < THIN_DEPTH_MUSD else "not_thin") for s in SYMBOLS if np.isfinite(DEPTH_TR[s])}
print("training depth ($M):", {s: round(v, 3) for s, v in DEPTH_TR.items()})
print("thin:", [s for s, g in GROUP.items() if g == "thin"])

def fit_terms(p, u, d, x):
    est = fit_t6(p, u, d, x)
    if est is None:
        return None
    t1, t2, t3 = est["phi"] * est["eps"], est["kappa"] ** 2 * est["eta"] * est["q"], (1 - est["a"]) * est["kappa"]
    est.update({"term_depth": t1, "term_momentum": t2, "term_reversion": t3,
                "A_ratio": (t1 + t2) / t3 - 1 if t3 > 1e-12 else np.nan})
    return est

def day_blocks5(s, d):
    df = PANEL[(s, d)]
    p, u, dd_, x, tot = prep(df, FLOW_SCALE[s])
    oi = df["oi"].ffill().to_numpy(); vol = df["vol"].to_numpy(); rows = []
    for hr in range(6, 24):
        t = hr * 60
        w, h, f = slice(t - 360, t), slice(t - 60, t), slice(t, min(t + 61, 1440))
        est = fit_terms(p[w], u[w], dd_[w], x[w])
        rows.append({"symbol": s, "date": d, "hour": hr,
                     **{k: (est[k] if est else np.nan) for k in ["C3", "A_ratio", "term_depth", "term_momentum", "term_reversion", "peak_ratio"]},
                     "rv_60": np.nanstd(np.diff(p[h])), "rv_360": np.nanstd(np.diff(p[w])), "abs_ret_60": abs(p[t - 1] - p[t - 60]),
                     "flow_imb_60": np.nansum(u[h]) / (np.nansum(np.abs(u[h])) + 1e-12),
                     "log_vol_60": np.log1p(np.nansum(vol[h])),
                     "log_depth1": np.log(tot[t - 1]) if np.isfinite(tot[t - 1]) and tot[t - 1] > 0 else np.nan,
                     "imbalance": dd_[t - 1],
                     "oi_chg_60": (oi[t - 1] / oi[t - 60] - 1) * 100 if np.isfinite(oi[t - 60]) and oi[t - 60] > 0 else np.nan,
                     "MAE_60": float(np.nanmax(np.abs(p[f] - p[t - 1])))})
    return rows

BLK5 = os.path.join(XDATA, f"blocks_p5_{len(SYMBOLS)}sym.parquet")
want = {(s, d) for (s, d) in OK if s in GROUP}
if os.path.exists(BLK5) and set(map(tuple, pd.read_parquet(BLK5)[["symbol", "date"]].drop_duplicates().to_numpy())) >= want:
    YB = pd.read_parquet(BLK5)
else:
    rows = []
    for i, (s, d) in enumerate(sorted(want)):
        rows += day_blocks5(s, d)
        if (i + 1) % 300 == 0:
            print(f"   blocks: {i+1}/{len(want)}")
    YB = pd.DataFrame(rows); YB.to_parquet(BLK5)
YB = YB.rename(columns={"C3": "C3_hat"})
YB["group"] = YB["symbol"].map(GROUP)
YB["split"] = np.where(YB["date"] <= TRAIN_END, "train", np.where(YB["date"] >= TEST_START, "test", "gap"))
print(YB.groupby(["group", "split"]).size().unstack())
record("Y3_groups", None, "audit", {"depth_train_musd": DEPTH_TR, "group": GROUP, "thin_cut_musd": THIN_DEPTH_MUSD,
       "blocks": YB.groupby(["group", "split"]).size().unstack().to_dict()})


## Pre-register, then test ONCE

In [ ]:
# ============================================================================
# CELL Y5 — PRE-REGISTRATION OF THE REPLICATION (run before Y6; commit prereg/P5_replication.json)
# The model, controls, outcome, inference and success rule are copied unchanged from Part 4 (X5).
# ============================================================================
CTRL = ["rv_60", "rv_360", "abs_ret_60", "flow_imb_60", "log_vol_60", "log_depth1", "imbalance", "oi_chg_60"]
PREREG_P5 = {
    "question": "Does the Part 4 thin-book result replicate on Binance USD-M symbols never used before?",
    "symbols": "drawn in cell Y2 (seeded) from thin-volume candidates not used in Part 4",
    "thin_rule": "median training depth within 1% of mid < 0.39 $M (Part 4 thin/mid boundary)",
    "unit": "symbol-hour blocks, hours 6-23 UTC; features [t-360, t); outcome (t, t+60]",
    "controls": CTRL + ["symbol fixed effects"],
    "primary": "group=thin, outcome=MAE_60, candidate=C3_hat winsorized at train 1%/99% (identical to Part 4 X6)",
    "secondary": ["thin: MAE_60 | A_ratio (unit-free)", "thin: MAE_60 | term_momentum", "thin: MAE_60 | term_depth",
                  "thin: MAE_60 | term_reversion", "not_thin: MAE_60 | C3_hat"],
    "train": "dates <= 2024-12-31", "test": "dates >= 2025-01-01",
    "model": "OLS with symbol fixed effects; tail = outcome above train 90th percentile",
    "inference": "Clark-West (Newey-West 6 lags); bootstrap 2,000 resampling dates for ΔR², ΔAUC",
    "success": "primary: Clark-West one-sided p < 0.05 AND bootstrap 95% CI of ΔR² excludes 0; secondary Holm-adjusted",
    "part4_hash": "5910b6b4d90aad457f03eb01d27cb0141933ff1d70dd93d85a98663f73e36020",
}
spec_txt5 = json.dumps(PREREG_P5, indent=2, sort_keys=True)
P5_HASH = hashlib.sha256(spec_txt5.encode()).hexdigest()
p5_path = os.path.join(PREREG, "P5_replication.json")
if not os.path.exists(p5_path):
    open(p5_path, "w").write(spec_txt5); print("pre-registration written:", p5_path)
elif hashlib.sha256(open(p5_path).read().encode()).hexdigest() != P5_HASH:
    print("!! WARNING: spec differs from the registered file. Any change after seeing results must be disclosed.")
print("spec SHA-256:", P5_HASH, "\nSave prereg/P5_replication.json somewhere dated BEFORE running Y6.")


In [ ]:
# ============================================================================
# CELL Y6 — REPLICATION TEST ON 2025-2026 (RUN ONCE; every run is counted)
# ============================================================================
assert os.path.exists(p5_path), "run Y5 first"
pth5 = os.path.join(PREREG, "test_set_unlocks_p5.json")
n_unl5 = (json.load(open(pth5))["n"] if os.path.exists(pth5) else 0) + 1
json.dump({"n": n_unl5, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "spec_sha256": P5_HASH}, open(pth5, "w"))
thin5 = YB[YB["group"] == "thin"]
primary5 = evaluate(thin5, "MAE_60", "C3_hat")
secondary5 = {"thin: MAE_60 | A_ratio": evaluate(thin5, "MAE_60", "A_ratio"),
              "thin: MAE_60 | term_momentum": evaluate(thin5, "MAE_60", "term_momentum"),
              "thin: MAE_60 | term_depth": evaluate(thin5, "MAE_60", "term_depth"),
              "thin: MAE_60 | term_reversion": evaluate(thin5, "MAE_60", "term_reversion"),
              "not_thin: MAE_60 | C3_hat": evaluate(YB[YB["group"] == "not_thin"], "MAE_60", "C3_hat")}
ps = sorted([(v["clark_west_p"] if np.isfinite(v.get("clark_west_p", np.nan)) else 1.0, k) for k, v in secondary5.items()])
for i, (p_, k) in enumerate(ps):
    secondary5[k]["holm_p"] = min(1.0, max(p2 * (len(ps) - j) for j, (p2, _) in enumerate(ps[:i + 1])))
show = lambda v: {k: (round(v[k], 5) if isinstance(v.get(k), float) else v.get(k)) for k in ["n_train", "n_test", "n_symbols", "delta_r2", "delta_r2_ci", "delta_auc_ci", "clark_west_p", "holm_p", "coef_candidate_train", "skipped"] if k in v}
print("PRIMARY (thin, new symbols):", show(primary5))
for k, v in secondary5.items():
    print(k, show(v))
ok5 = bool(np.isfinite(primary5.get("clark_west_p", np.nan)) and primary5["clark_west_p"] < 0.05 and primary5["delta_r2_ci"][0] > 0)
record("Y6_replication_G2", ok5, "real-data", {"primary": primary5, "secondary": secondary5, "prereg_sha256": P5_HASH,
       "test_set_unlocks": n_unl5}, criterion=PREREG_P5["success"],
       note="Out-of-sample replication on new symbols. Report whatever it shows; unlocks > 1 must be disclosed.")


## Descriptives and download

In [ ]:
# ============================================================================
# CELL Y7 — DESCRIPTIVES (training only) AND DOWNLOAD
# ============================================================================
trn = YB[YB["split"] == "train"]
desc = {}
for g in ["thin", "not_thin"]:
    T = trn[trn["group"] == g]
    if len(T):
        desc[g] = {"blocks": len(T), "share_C3_pos": float((T["C3_hat"] > 0).mean()),
                   **{f"median_{c}": float(T[c].median()) for c in ["C3_hat", "A_ratio", "term_depth", "term_momentum", "term_reversion"]},
                   "corr_MAE": {c: float(T[[c, "MAE_60"]].corr(method="spearman").iloc[0, 1]) for c in ["C3_hat", "A_ratio", "term_depth", "term_momentum", "term_reversion"]}}
print(json.dumps(desc, indent=1))
record("Y7_descriptives_train", None, "real-data", desc, note="Training period only: which term of C3 moves with tail risk.")
zp = os.path.join(BASE, "p5_results.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in glob.glob(os.path.join(RESULTS, "Y*.json")) + [os.path.join(RESULTS, "p5_manifest_sha256.csv"), p5_path, pth5]:
        if os.path.exists(fp):
            z.write(fp, os.path.relpath(fp, BASE))
print("zipped ->", zp)
if IN_COLAB:
    from google.colab import files
    files.download(zp)


## How to read Y6
| Result | Meaning for Paper 2 |
|---|---|
| Primary passes (CW p < 0.05 and ΔR² CI > 0) | **Replicated out of sample on new symbols**: the strongest possible evidence; Paper 2 → about 8.5 |
| Primary fails, same sign | Effect real but smaller or noisier than in Part 4; report both honestly; about 7.5–8 |
| Opposite sign | Part 4's result does not generalize; Paper 2 → about 7 |
| A_ratio passes | the unit-free version works; this answers the scale concern from X6r |
| term_momentum passes but term_depth does not | mechanism = momentum feedback, consistent with X8 |

Send me `p5_results.zip`.

## Y8 — post-hoc: is C3 just price impact? (decided after Y6)
**Do NOT re-run Y6.** In a fresh session run Y0, Y1, Y2, Y3, Y4, Y5 (all read from the Drive cache), skip Y6 and Y7, then run Y8.
Y8 adds four price-impact controls (κ̂, κ̂², Kyle's λ, Amihud) and asks whether C3 still forecasts. It uses a separate counter, so Y6's unlock count stays at 1.
Expected time: about 10–20 minutes. Send me `p5_kappa.zip`.

In [ ]:
# ============================================================================
# CELL Y8 — POST-HOC: IS C3 JUST PRICE IMPACT? (run after Y0–Y4; does NOT touch Y6's counter)
# Decided AFTER seeing Y6. Report as post-hoc robustness. Counted in prereg/robustness_evals_p5.json.
# Both terms that carried Y6 (momentum κ²ηq and reversion (1-a)κ) contain the impact coefficient κ.
# If C3 only measures price impact, adding impact controls should erase its forecasting gain.
# Impact controls (all from the same past 6 h / 1 h, no look-ahead), winsorized 1/99 on training data:
#   kappa_hat   κ from the same T6 fit that produces C3
#   kappa_sq    κ²
#   kyle_lam    univariate slope of Δp on signed flow, past 6 h
#   amihud_60   |60-min return| / 60-min volume, logged
# Checks (thin group, new symbols, same evaluation code as Y6):
#   K0 baseline C3 (must match Y6)          K1 C3 | + kappa_hat, kappa_sq
#   K2 C3 | + kyle_lam, amihud_60            K3 C3 | all four impact controls
#   K4 term_momentum | all four              K5 term_reversion | all four
#   K6 kappa_hat alone as the candidate (does raw impact forecast?)
# Reading: C3 is more than impact if K3 keeps CW p < 0.05 with ΔR² CI above 0.
# ============================================================================
BLK8 = os.path.join(XDATA, f"blocks_p5_kappa_{len(SYMBOLS)}sym.parquet")
thin_syms = sorted(s for s, g in GROUP.items() if g == "thin")
want8 = {(s, d) for (s, d) in OK if s in thin_syms}
assert want8, "no thin symbol-days: run Y0-Y4 first"
assert "CTRL" in globals(), "run Y5 first (defines CTRL)"

def day_blocks8(s, d):
    df = PANEL[(s, d)]
    p, u, dd_, x, tot = prep(df, FLOW_SCALE[s])
    vol = df["vol"].to_numpy(); dp = np.diff(p, prepend=np.nan); rows = []
    for hr in range(6, 24):
        t = hr * 60
        w, h = slice(t - 360, t), slice(t - 60, t)
        est = fit_t6(p[w], u[w], dd_[w], x[w])
        uu, yy = u[w][1:], dp[w][1:]; ok = np.isfinite(uu) & np.isfinite(yy)
        kl = float(np.polyfit(uu[ok], yy[ok], 1)[0]) if ok.sum() > 60 and np.std(uu[ok]) > 0 else np.nan
        v60 = np.nansum(vol[h])
        rows.append({"symbol": s, "date": d, "hour": hr,
                     "kappa_hat": est["kappa"] if est else np.nan, "kyle_lam": kl,
                     "amihud_60": np.log(abs(p[t - 1] - p[t - 60]) / v60 + 1e-9) if v60 > 0 and np.isfinite(p[t - 60]) else np.nan})
    return rows

if os.path.exists(BLK8) and set(map(tuple, pd.read_parquet(BLK8)[["symbol", "date"]].drop_duplicates().to_numpy())) >= want8:
    K8 = pd.read_parquet(BLK8)
else:
    rows = []
    for i, (s, d) in enumerate(sorted(want8)):
        rows += day_blocks8(s, d)
        if (i + 1) % 200 == 0:
            print(f"   impact blocks: {i+1}/{len(want8)}")
    K8 = pd.DataFrame(rows); K8.to_parquet(BLK8)

YK = YB[YB["group"] == "thin"].merge(K8, on=["symbol", "date", "hour"], how="left")
YK["kappa_sq"] = YK["kappa_hat"] ** 2
IMPACT = ["kappa_hat", "kappa_sq", "kyle_lam", "amihud_60"]
for c in IMPACT:                                    # winsorize controls with training quantiles only
    lo, hi = YK.loc[YK["split"] == "train", c].quantile([0.01, 0.99])
    YK[c] = YK[c].clip(lo, hi)
print("impact-control coverage:", {c: round(float(YK[c].notna().mean()), 3) for c in IMPACT})
print("train correlations with C3_hat:", YK[YK["split"] == "train"][["C3_hat"] + IMPACT].corr()["C3_hat"].round(3).to_dict())

def evaluate_ctrl(sub, outcome, cand, extra):
    global CTRL
    _saved = CTRL
    try:
        CTRL = _saved + extra
        return evaluate(sub, outcome, cand)
    finally:
        CTRL = _saved

checks8 = {
    "K0_baseline_C3":            evaluate_ctrl(YK, "MAE_60", "C3_hat", []),
    "K1_C3_plus_kappa":          evaluate_ctrl(YK, "MAE_60", "C3_hat", ["kappa_hat", "kappa_sq"]),
    "K2_C3_plus_kyle_amihud":    evaluate_ctrl(YK, "MAE_60", "C3_hat", ["kyle_lam", "amihud_60"]),
    "K3_C3_plus_all_impact":     evaluate_ctrl(YK, "MAE_60", "C3_hat", IMPACT),
    "K4_momentum_plus_all":      evaluate_ctrl(YK, "MAE_60", "term_momentum", IMPACT),
    "K5_reversion_plus_all":     evaluate_ctrl(YK, "MAE_60", "term_reversion", IMPACT),
    "K6_kappa_alone":            evaluate_ctrl(YK, "MAE_60", "kappa_hat", []),
}
ps = sorted([(v["clark_west_p"] if np.isfinite(v.get("clark_west_p", np.nan)) else 1.0, k) for k, v in checks8.items() if k != "K0_baseline_C3"])
for i, (p_, k) in enumerate(ps):
    checks8[k]["holm_p"] = min(1.0, max(p2 * (len(ps) - j) for j, (p2, _) in enumerate(ps[:i + 1])))

pth8 = os.path.join(PREREG, "robustness_evals_p5.json")
n_rob8 = (json.load(open(pth8))["n"] if os.path.exists(pth8) else 0) + 1
json.dump({"n": n_rob8, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, open(pth8, "w"))

rows = []
for k, v in checks8.items():
    if v.get("skipped"):
        rows.append({"check": k, "note": v["skipped"]}); continue
    rows.append({"check": k, "n_test": v["n_test"], "dR2_pts": round(100 * v["delta_r2"], 3),
                 "CI_pts": [round(100 * c, 3) for c in v["delta_r2_ci"]],
                 "dAUC_CI": [round(c, 4) for c in v["delta_auc_ci"]],
                 "CW_p": round(v["clark_west_p"], 5), "Holm_p": round(v.get("holm_p", np.nan), 5),
                 "coef": round(v["coef_candidate_train"], 2)})
print(pd.DataFrame(rows).to_string(index=False))
k3 = checks8["K3_C3_plus_all_impact"]
survives = bool(not k3.get("skipped") and k3["clark_west_p"] < 0.05 and k3["delta_r2_ci"][0] > 0)
print("\nC3 survives all impact controls:", survives)
record("Y8_kappa_control_posthoc", survives, "real-data",
       {"checks": checks8, "impact_controls": IMPACT, "robustness_evaluations": n_rob8,
        "train_corr_with_C3": YK[YK["split"] == "train"][["C3_hat"] + IMPACT].corr()["C3_hat"].to_dict()},
       criterion="K3: CW p < 0.05 and ΔR² CI above 0 with all four impact controls added",
       note="POST-HOC. Report separately from the pre-registered Y6 result.")
_zp8 = os.path.join(BASE, "p5_kappa.zip")
with zipfile.ZipFile(_zp8, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in [os.path.join(RESULTS, "Y8_kappa_control_posthoc.json"), pth8]:
        z.write(fp, os.path.relpath(fp, BASE))
print("zipped ->", _zp8)
if IN_COLAB:
    from google.colab import files
    files.download(_zp8)
